# Detector images, all temperatures

Same vertical crop and jet scale as the 2×3 detector montage, now all nine temperatures in a 3×3 grid. Temperature labels only, colored with the locked turbo scale.


In [ ]:
import sys
sys.path.append('../src/')

from pathlib import Path
import json

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

try:
    import hdf5plugin  # noqa: F401
except ImportError:
    pass

from postprocess import (
    FOUR_PEAK_SCANS,
    SINGLE_PEAK_SCANS,
    temperature_cmap,
    temperature_color,
    temperature_norm,
)


In [ ]:
def apply_paper_style():
    helvetica = font_manager.FontProperties(family='Helvetica')
    font_manager.findfont(helvetica, fallback_to_default=False)
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Helvetica', 'Arial'],
        'font.size': 9,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.dpi': 600,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.02,
    })


apply_paper_style()

cmap = temperature_cmap()
norm = temperature_norm()

CROP_ROWS = (646, 1163)
CROP_COLS = (880, 1460)
NFRAMES = 100
HOT_PIXEL = 50.0
VMIN, VMAX = 0.0, 50.0

config_path = Path('../configs/config_B10.json')
config = json.loads(config_path.read_text()) if config_path.exists() else {}
APS_BASE = Path(config['Base']) if config.get('Base') else Path('/Users/eriklamb/data/APS/8_ID_E/Na2B10H10')

SCANS = tuple(FOUR_PEAK_SCANS) + tuple(SINGLE_PEAK_SCANS)


In [ ]:
def find_raw(T, scan):
    folder = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001'
    hits = sorted(folder.glob('*_001.h5'))
    if not hits:
        raise FileNotFoundError(f'no raw detector file for T={T} scan={scan}')
    return hits[0]


def load_sum(T, scan, nframes=NFRAMES):
    path = find_raw(T, scan)
    r0, r1 = CROP_ROWS
    c0, c1 = CROP_COLS
    with h5py.File(path, 'r') as f:
        ntot = f['entry/data/data'].shape[0]
        det = np.asarray(f['entry/data/data'][:min(nframes, ntot), r0:r1, c0:c1], dtype=np.float64)
    det[det > HOT_PIXEL] = 0.0
    return det.sum(axis=0)


images = []
for T, scan in SCANS:
    img = load_sum(T, scan)
    images.append((T, img))
    print(f'{T:g} K  scan {scan}  {img.shape[0]}x{img.shape[1]}  max={img.max():.0f}')


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(7.4, 6.4))
for ax, (T, img) in zip(axes.ravel(), images):
    ax.imshow(img, cmap='jet', vmin=VMIN, vmax=VMAX, interpolation='nearest')
    ax.set_axis_off()
    ax.text(
        0.04, 0.96, f'{T:g} K',
        transform=ax.transAxes, ha='left', va='top', fontsize=11,
        color=temperature_color(T, cmap=cmap, norm=norm),
        bbox=dict(boxstyle='square,pad=0.12', facecolor='white', edgecolor='none', alpha=1.0),
    )

fig.subplots_adjust(wspace=0.04, hspace=0.04)
out = Path('../figures')
out.mkdir(exist_ok=True)
fig.savefig(out / 'detector_images.pdf')
fig.savefig(out / 'detector_images.png')
print('saved', out / 'detector_images.pdf')
